In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create the project folder structure
import os
PROJECT_DIR = '/content/drive/MyDrive/NLP_Project/member3_bigram'
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder ready:", PROJECT_DIR)

Mounted at /content/drive
Project folder ready: /content/drive/MyDrive/NLP_Project/member3_bigram


In [4]:
import re, pickle, itertools
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
PROJECT_DIR = '/content/drive/MyDrive/NLP_Project/member3_bigram'
nltk.download('movie_reviews')

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


True

In [6]:
import pickle
import numpy as np
import pandas as pd

PROJECT_DIR = '/content/drive/MyDrive/NLP_Project/member3_bigram'

# If Drive is not mounted in this session, mount it first
from google.colab import drive
drive.mount('/content/drive')

with open(f'{PROJECT_DIR}/bigram_model.pkl', 'rb') as f:
    final_model = pickle.load(f)

print(final_model)

Mounted at /content/drive
Pipeline(steps=[('vec', CountVectorizer(binary=True, ngram_range=(1, 2))),
                ('nb', MultinomialNB())])


In [9]:
# TEMPORARY split: used only until Member 1's final train_test_split.csv is ready.
rows = []
for fid in movie_reviews.fileids():
    rows.append({
        'review_id': fid,                       # e.g. 'pos/cv000_29590.txt'
        'text': movie_reviews.raw(fid),
        'label': movie_reviews.categories(fid)[0]   # 'pos' or 'neg' in NLTK -> mapped below
    })
df = pd.DataFrame(rows)
df['label'] = df['label'].map({'pos': 'positive', 'neg': 'negative'})

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)
print(train_df.shape, test_df.shape)
print(train_df['label'].value_counts())
print(test_df['label'].value_counts())

train_df.to_csv(f'{PROJECT_DIR}/TEMP_train.csv', index=False)
test_df.to_csv(f'{PROJECT_DIR}/TEMP_test.csv', index=False)

(1600, 3) (400, 3)
label
positive    800
negative    800
Name: count, dtype: int64
label
negative    200
positive    200
Name: count, dtype: int64


In [10]:
sample = ["the movie was not good", "the movie was very good"]

uni = CountVectorizer(ngram_range=(1, 1))
uni.fit(sample)
print("Unigrams:", list(uni.get_feature_names_out()))

bi = CountVectorizer(ngram_range=(1, 2))
bi.fit(sample)
print("Unigram+Bigram:", list(bi.get_feature_names_out()))
# Notice 'not good' appears as ONE feature: this is how bigrams capture simple negation.

Unigrams: ['good', 'movie', 'not', 'the', 'very', 'was']
Unigram+Bigram: ['good', 'movie', 'movie was', 'not', 'not good', 'the', 'the movie', 'very', 'very good', 'was', 'was not', 'was very']


In [ ]:
# Reference only. Member 2 owns the official unigram model.
X_train, y_train = train_df['text'], train_df['label']
X_test, y_test = test_df['text'], test_df['label']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

ref_pipe = Pipeline([
    ('vec', CountVectorizer(ngram_range=(1, 1))),
    ('nb', MultinomialNB(alpha=1.0)),
])
ref_scores = cross_val_score(ref_pipe, X_train, y_train, cv=cv, scoring='accuracy')
print("Unigram reference CV accuracy: %.4f" % ref_scores.mean())

Unigram reference CV accuracy: 0.8225


In [ ]:
bi_pipe = Pipeline([
    ('vec', CountVectorizer(ngram_range=(1, 2))),
    ('nb', MultinomialNB(alpha=1.0)),
])
bi_scores = cross_val_score(bi_pipe, X_train, y_train, cv=cv, scoring='accuracy')
print("Unigram+Bigram CV accuracy: %.4f" % bi_scores.mean())

Unigram+Bigram CV accuracy: 0.8344


In [ ]:
# ngram_range stays (1, 2): the one-change comparison is only unigram vs unigram+bigram.
# Here we vary secondary settings, which is the "feature extraction settings" experiment.
grid = {
    'min_df': [1, 2, 5],
    'stop_words': [None, 'english'],
    'binary': [False, True],
    'alpha': [0.1, 0.5, 1.0],
}

results = []
for min_df, stop_words, binary, alpha in itertools.product(*grid.values()):
    pipe = Pipeline([
        ('vec', CountVectorizer(ngram_range=(1, 2), min_df=min_df,
                                stop_words=stop_words, binary=binary)),
        ('nb', MultinomialNB(alpha=alpha)),
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    results.append({'min_df': min_df, 'stop_words': stop_words, 'binary': binary,
                    'alpha': alpha, 'cv_mean': scores.mean(), 'cv_std': scores.std()})

exp_df = pd.DataFrame(results).sort_values('cv_mean', ascending=False).reset_index(drop=True)
exp_df.to_csv(f'{PROJECT_DIR}/bigram_experiment_results.csv', index=False)
exp_df.head(10)

,min_df,stop_words,binary,alpha,cv_mean,cv_std
0,1,None,True,1.0,0.864375,0.015637
1,1,None,True,0.5,0.863125,0.018687
2,5,None,True,1.0,0.861875,0.012406
3,2,None,True,1.0,0.861875,0.015487
4,2,None,True,0.5,0.861250,0.016840
5,5,None,True,0.5,0.858750,0.017500
6,5,None,True,0.1,0.856250,0.018435
7,1,None,True,0.1,0.856250,0.016178
8,2,None,True,0.1,0.853750,0.012562
9,5,english,True,0.5,0.851875,0.017410


In [ ]:
best = exp_df.iloc[0]
best_sw = None if pd.isna(best['stop_words']) else best['stop_words']
print("Best settings:\n", best)

final_model = Pipeline([
    ('vec', CountVectorizer(ngram_range=(1, 2), min_df=int(best['min_df']),
                            stop_words=best_sw, binary=bool(best['binary']))),
    ('nb', MultinomialNB(alpha=float(best['alpha']))),
])
final_model.fit(X_train, y_train)

# TEST SET: used once, here only.
y_pred = final_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
print("Test accuracy: %.4f | macro P: %.4f | R: %.4f | F1: %.4f" % (acc, p, r, f1))
print(confusion_matrix(y_test, y_pred, labels=['negative', 'positive']))

# Unigram reference on the same test set, for context only
ref_pipe.fit(X_train, y_train)
print("Unigram reference test accuracy: %.4f" % accuracy_score(y_test, ref_pipe.predict(X_test)))

Best settings:
 min_df               1
stop_words        None
binary            True
alpha              1.0
cv_mean       0.864375
cv_std        0.015637
Name: 0, dtype: object
Test accuracy: 0.8600 | macro P: 0.8629 | R: 0.8600 | F1: 0.8597
[[163  37]
 [ 19 181]]
Unigram reference test accuracy: 0.8175


In [ ]:
pred_df = pd.DataFrame({
    'review_id': test_df['review_id'].values,
    'true_label': y_test.values,
    'predicted_label': y_pred,
})
pred_df.to_csv(f'{PROJECT_DIR}/bigram_predictions.csv', index=False)

with open(f'{PROJECT_DIR}/bigram_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)   # one file holds vectorizer + classifier

print(pred_df.head())
print(pred_df.shape)

             review_id true_label predicted_label
0  neg/cv748_14044.txt   negative        negative
1   neg/cv020_9234.txt   negative        negative
2  pos/cv088_24113.txt   positive        positive
3  neg/cv466_20092.txt   negative        positive
4  neg/cv491_12992.txt   negative        negative
(400, 3)


In [7]:
vec = final_model.named_steps['vec']
nb = final_model.named_steps['nb']
names = vec.get_feature_names_out()
log_ratio = nb.feature_log_prob_[list(nb.classes_).index('positive')] - \
            nb.feature_log_prob_[list(nb.classes_).index('negative')]

order = np.argsort(log_ratio)
def show(idx, title):
    print(f"\n{title}")
    for i in idx:
        print(f"  {names[i]:30s} {log_ratio[i]:+.3f}")

show(order[::-1][:20], "Top 20 POSITIVE features (words + phrases)")
show(order[:20], "Top 20 NEGATIVE features (words + phrases)")

is_bigram = np.array([' ' in n for n in names])
bi_idx = np.where(is_bigram)[0]
bi_order = bi_idx[np.argsort(log_ratio[bi_idx])]
show(bi_order[::-1][:15], "Top 15 POSITIVE bigrams only")
show(bi_order[:15], "Top 15 NEGATIVE bigrams only")


Top 20 POSITIVE features (words + phrases)
  is excellent                   +2.776
  is terrific                    +2.684
  rousing                        +2.651
  dread                          +2.582
  characters with                +2.582
  mulan                          +2.508
  scenes as                      +2.508
  very effective                 +2.508
  best movie                     +2.508
  she loves                      +2.508
  strengths                      +2.508
  for change                     +2.428
  american history               +2.428
  missteps                       +2.341
  director james                 +2.341
  is rare                        +2.341
  fits the                       +2.341
  meryl                          +2.341
  lovingly                       +2.341
  the critics                    +2.341

Top 20 NEGATIVE features (words + phrases)
  this mess                      -3.276
  and boring                     -3.053
  is terrible                   

In [8]:
with open(f'{PROJECT_DIR}/bigram_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

tests = ["This movie was not good at all, a total waste of time.",
         "A brilliant film with wonderful acting and a moving story."]
print(loaded.predict(tests))

chk = pd.read_csv(f'{PROJECT_DIR}/bigram_predictions.csv')
assert list(chk.columns) == ['review_id', 'true_label', 'predicted_label']
print("CSV format OK, rows:", len(chk))

['negative' 'positive']
CSV format OK, rows: 400
